In [1]:
import numpy as np, pandas as pd, joblib, json
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)
from xgboost import XGBClassifier

ROOT = Path('..')
FIG = ROOT/'reports'/'figures'; FIG.mkdir(parents=True, exist_ok=True)
TAB = ROOT/'reports'/'tables';  TAB.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ROOT/'data'/'processed'/'customer_features_labeled.csv')
model  = joblib.load(ROOT/'models'/'logistic_churn_model.pkl')
scaler = joblib.load(ROOT/'models'/'scaler.pkl')
cfg    = joblib.load(ROOT/'models'/'model_config.pkl')
F, THR = cfg['features'], cfg['threshold']    
X, y = df[F], df['is_churn']

In [2]:
X_tv, X_te, y_tv, y_te = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_tr, X_va, y_tr, y_va = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)
assert (len(X_tr), len(X_va), len(X_te)) == (1594, 532, 532)

In [3]:
rf  = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=4,
                             class_weight='balanced', random_state=42, n_jobs=-1).fit(X_tr, y_tr)
xgb = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03,
                    random_state=42, eval_metric='logloss').fit(X_tr, y_tr)
p_va = {'Logistic Regression': model.predict_proba(scaler.transform(X_va))[:, 1],
        'RF_3': rf.predict_proba(X_va)[:, 1],
        'XGB_2': xgb.predict_proba(X_va)[:, 1]}

In [4]:
def metrics(yt, p, t):
    h = (p >= t).astype(int)
    return dict(Accuracy=accuracy_score(yt, h), Precision=precision_score(yt, h, zero_division=0),
                Recall=recall_score(yt, h), F1=f1_score(yt, h), ROC_AUC=roc_auc_score(yt, p))

# (a) cấu hình cuối như Draft: LR@0.35, RF_3@0.50, XGB_2@0.50
final_val = pd.DataFrame([{'Model': n, 'Threshold': t, **metrics(y_va, p_va[k], t)}
    for n, k, t in [('Logistic Regression', 'Logistic Regression', THR),
                    ('RF_3', 'RF_3', 0.5), ('XGB_2', 'XGB_2', 0.5)]])
# (b) dò ngưỡng cho cả ba
sweep = pd.DataFrame([{'Model': n, 'Threshold': round(t, 2), **metrics(y_va, p, t)}
    for n, p in p_va.items() for t in np.arange(0.15, 0.65, 0.05)])
best = sweep.loc[sweep.groupby('Model')['F1'].idxmax()]     # ngưỡng cho F1 tốt nhất của mỗi mô hình
final_val.round(4).to_csv(TAB/'validation_final.csv', index=False)
sweep.round(4).to_csv(TAB/'threshold_sweep_all.csv', index=False)
display(final_val.round(4)); display(best.round(4))

,Model,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.35,0.7030,0.5064,0.7391,0.6010,0.7631
1,RF_3,0.50,0.6898,0.4918,0.7453,0.5926,0.7791
2,XGB_2,0.50,0.7350,0.5714,0.4969,0.5316,0.7675


,Model,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC
4,Logistic Regression,0.35,0.7030,0.5064,0.7391,0.6010,0.7631
15,RF_3,0.40,0.6523,0.4600,0.8571,0.5987,0.7791
21,XGB_2,0.20,0.6278,0.4420,0.8758,0.5875,0.7675


In [5]:
plt.figure(figsize=(6, 5))
for n, p in p_va.items():
    fpr, tpr, _ = roc_curve(y_va, p); plt.plot(fpr, tpr, label=f'{n} (AUC = {roc_auc_score(y_va, p):.3f})')
plt.plot([0, 1], [0, 1], '--', color='gray'); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('Đường cong ROC trên tập validation'); plt.legend(frameon=False); plt.tight_layout()
plt.savefig(FIG/'roc_validation_3models.png', dpi=200); plt.close()

lr = sweep[sweep.Model == 'Logistic Regression']
plt.figure(figsize=(6.5, 4))
for c in ['Precision', 'Recall', 'F1']: plt.plot(lr.Threshold, lr[c], marker='o', label=c)
plt.axvline(THR, ls='--', color='gray'); plt.xlabel('Decision threshold'); plt.legend(frameon=False)
plt.title('Precision, Recall, F1 của Logistic Regression theo ngưỡng (validation)'); plt.tight_layout()
plt.savefig(FIG/'threshold_curve_lr.png', dpi=200); plt.close()

In [6]:
p_te = model.predict_proba(scaler.transform(X_te))[:, 1]
test_res = metrics(y_te, p_te, THR); print({k: round(v, 4) for k, v in test_res.items()})
cm = confusion_matrix(y_te, (p_te >= THR).astype(int)); print(cm)
tn, fp, fn, tp = cm.ravel()
extra = dict(Specificity=tn/(tn+fp), NPV=tn/(tn+fn), FPR=fp/(fp+tn), FNR=fn/(fn+tp),
             Flagged_rate=(tp+fp)/cm.sum(), Base_rate=(tp+fn)/cm.sum(),
             Lift=(tp/(tp+fp))/((tp+fn)/cm.sum()), Majority_acc=(tn+fp)/cm.sum())
print({k: round(v, 4) for k, v in extra.items()})

ConfusionMatrixDisplay(cm, display_labels=['Không churn', 'Churn']).plot(cmap='Blues', values_format='d')
plt.title('Ma trận nhầm lẫn trên tập test (ngưỡng 0,35)'); plt.tight_layout()
plt.savefig(FIG/'confusion_matrix_test.png', dpi=200); plt.close()

rng = np.random.default_rng(42); yt = y_te.values; bs = []
for _ in range(2000):
    i = rng.integers(0, len(yt), len(yt)); h = (p_te[i] >= THR).astype(int)
    bs.append([precision_score(yt[i], h, zero_division=0), recall_score(yt[i], h),
               f1_score(yt[i], h), roc_auc_score(yt[i], p_te[i])])
ci = pd.DataFrame(np.percentile(bs, [2.5, 97.5], axis=0), index=['2.5%', '97.5%'],
                  columns=['Precision', 'Recall', 'F1', 'ROC_AUC']); display(ci.round(3))

{'Accuracy': 0.6898, 'Precision': 0.4915, 'Recall': 0.7205, 'F1': 0.5844, 'ROC_AUC': 0.7679}
[[251 120]
 [ 45 116]]
{'Specificity': np.float64(0.6765), 'NPV': np.float64(0.848), 'FPR': np.float64(0.3235), 'FNR': np.float64(0.2795), 'Flagged_rate': np.float64(0.4436), 'Base_rate': np.float64(0.3026), 'Lift': np.float64(1.6242), 'Majority_acc': np.float64(0.6974)}


,Precision,Recall,F1,ROC_AUC
2.5%,0.430,0.648,0.525,0.723
97.5%,0.553,0.784,0.637,0.807


In [7]:
summary = {'validation': final_val.round(4).to_dict('records'),
           'best_threshold_per_model': best.round(4).to_dict('records'),
           'test_logistic_regression': {k: round(v, 4) for k, v in test_res.items()},
           'test_confusion_matrix': cm.tolist()}
json.dump(summary, open(ROOT/'models'/'metrics_summary.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)

In [8]:
df['churn_probability'] = model.predict_proba(scaler.transform(X))[:, 1]
df['predicted_churn']   = (df['churn_probability'] >= THR).astype(int)
df['risk_level'] = pd.cut(df['churn_probability'], [-0.001, THR, 0.60, 1.0], labels=['Low', 'Medium', 'High'])
df['split'] = 'train'; df.loc[X_va.index, 'split'] = 'val'; df.loc[X_te.index, 'split'] = 'test'

risk_all  = df.groupby('risk_level', observed=True).agg(So_khach=('is_churn', 'size'), Churn_thuc_te=('is_churn', 'mean'),
              Monetary_TB=('Monetary', 'mean'), Recency_TB=('Recency', 'mean'), Frequency_TB=('Frequency', 'mean')).round(3)
risk_test = df[df.split == 'test'].groupby('risk_level', observed=True).agg(
              So_khach=('is_churn', 'size'), Churn_thuc_te=('is_churn', 'mean')).round(3)
display(risk_all); display(risk_test)
risk_all.to_csv(TAB/'risk_level_all.csv'); risk_test.to_csv(TAB/'risk_level_test.csv')

cols = ['Customer ID', 'churn_probability', 'risk_level', 'predicted_churn', 'is_churn', 'split'] + F + ['Country']
df[cols].to_csv(ROOT/'data'/'processed'/'churn_predictions.csv', index=False)

plt.figure(figsize=(6.5, 4)); plt.hist(df.churn_probability, bins=40)
for v in (THR, 0.60): plt.axvline(v, ls='--', color='gray')
plt.xlabel('Churn probability'); plt.ylabel('Số khách hàng'); plt.title('Phân bố xác suất churn của 2.658 khách hàng')
plt.tight_layout(); plt.savefig(FIG/'churn_probability_distribution.png', dpi=200); plt.close()

,So_khach,Churn_thuc_te,Monetary_TB,Recency_TB,Frequency_TB
risk_level,,,,,
Low,1487,0.139,4122.619,39.956,8.557
Medium,973,0.479,564.611,77.763,1.937
High,198,0.662,308.929,142.586,1.227


,So_khach,Churn_thuc_te
risk_level,,
Low,296,0.152
Medium,196,0.439
High,40,0.750
